# PaLI-Gemma 2 Fine-Tuning

In [ ]:
!pip install -q -U datasets bitsandbytes peft git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.0/502.0 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.0.0.dev0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires p

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: harisankar1320 (harisankar1320-german-aerospace-center-dlr) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import torch
device = torch.cuda.is_available()
print(device)

True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import json

IMAGES_DIR = "/content/drive/MyDrive/ALL IN_pg/New Work for Journal /images"
TRAIN_JSON = Path("/content/drive/MyDrive/ALL IN_pg/New Work for Journal /Paligemma split new/train_new.jsonl")
VAL_JSON   = Path("/content/drive/MyDrive/ALL IN_pg/New Work for Journal /Paligemma split new/val_new.jsonl")
TEST_JSON  = Path("/content/drive/MyDrive/ALL IN_pg/New Work for Journal /Paligemma split new/test_new.jsonl")

### The below cell prepares the dataset loader. It reads the JSONL files, finds the matching image for each sample, loads the image, and returns the instruction prefix and target output. This makes the remote sensing dataset compatible with PyTorch training.

In [ ]:
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image
import json

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

def _build_image_index(images_dir: str):
    """Case-insensitive index: basename -> absolute Path (first hit wins)."""
    images_dir = Path(images_dir)
    index = {}
    for p in images_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            key = p.name.lower()
            if key not in index:
                index[key] = p
    return index

class JSONLDataset(Dataset):
    def __init__(self, jsonl_path: str, images_dir: str, drop_missing: bool = True):
        self.jsonl_path = Path(jsonl_path)
        self.images_dir = Path(images_dir)
        if not self.jsonl_path.is_file():
            raise FileNotFoundError(f"JSONL not found: {self.jsonl_path}")
        if not self.images_dir.is_dir():
            raise NotADirectoryError(f"Images dir not found: {self.images_dir}")

        self._img_index = _build_image_index(self.images_dir)
        self.samples = []

        bad_json = 0
        missing_img = 0

        with self.jsonl_path.open("r", encoding="utf-8") as f:
            for ln, line in enumerate(f, 1):
                s = line.strip()
                if not s:
                    continue
                try:
                    obj = json.loads(s)
                except Exception:
                    bad_json += 1
                    continue

                img_name = Path(obj.get("image", "")).name
                prefix   = obj.get("prefix", "")
                suffix   = obj.get("suffix", "")

                if not img_name:
                    if drop_missing:
                        continue
                    else:
                        img_path = None
                else:
                    img_path = self._img_index.get(img_name.lower())

                if img_path is None:
                    missing_img += 1
                    if drop_missing:
                        continue

                self.samples.append({
                    "image_name": img_name,
                    "image_path": img_path,
                    "prefix": prefix,
                    "suffix": suffix,
                })

        print(f"[JSONLDataset] {self.jsonl_path.name}: kept={len(self.samples)} | "
              f"missing_img={missing_img} | bad_json={bad_json}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["image_path"]).convert("RGB")
        return {
            "image": img,
            "prefix": s["prefix"],
            "suffix": s["suffix"],
            "image_path": str(s["image_path"]),
        }

# Instantiate datasets (names from your snippet)
train_ds = JSONLDataset(TRAIN_JSON, IMAGES_DIR, drop_missing=True)
val_ds   = JSONLDataset(VAL_JSON,   IMAGES_DIR, drop_missing=True)
test_ds  = JSONLDataset(TEST_JSON,  IMAGES_DIR, drop_missing=True)

[JSONLDataset] train_new.jsonl: kept=13240 | missing_img=0 | bad_json=0
[JSONLDataset] val_new.jsonl: kept=1656 | missing_img=0 | bad_json=0
[JSONLDataset] test_new.jsonl: kept=1656 | missing_img=0 | bad_json=0


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# for name, _ in model.named_modules():
#     if "projector" in name.lower():
#         print(name)


## This cell loads the pretrained PaLI-Gemma 2 model and processor. It also applies 4-bit quantization and LoRA configuration, allowing the large model to be fine-tuned with lower GPU memory usage.

In [ ]:
from transformers import BitsAndBytesConfig, PaliGemmaForConditionalGeneration
from peft import get_peft_model, LoraConfig
from transformers import PaliGemmaForConditionalGeneration
import torch
from transformers import PaliGemmaProcessor

model_id ="google/paligemma2-3b-mix-448"
use_qlora = True
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True,
  bnb_4bit_compute_dtype=torch.bfloat16,
)
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj", "multi_modal_projector.linear"],
    task_type="CAUSAL_LM",
)

model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, device_map="auto", quantization_config=bnb_config, torch_dtype = torch.bfloat16)

for param in model.vision_tower.parameters():
    param.requires_grad = False

# for param in model.multi_modal_projector.parameters():
#     param.requires_grad = True

model.config.use_cache = False

model = get_peft_model(model, lora_config)

model.gradient_checkpointing_enable()

processor = PaliGemmaProcessor.from_pretrained(model_id)

model.print_trainable_parameters()


config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/75.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.07G [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.6M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

trainable params: 47,616,000 || all params: 3,080,743,152 || trainable%: 1.5456


In [ ]:
for name, param in model.named_parameters():
    if "projector" in name:
        print(name, param.requires_grad)

base_model.model.model.multi_modal_projector.linear.weight False
base_model.model.model.multi_modal_projector.linear.bias False


### This cell defines helper functions for task detection, location-token cleaning, and batch preparation. It formats segmentation, detection, captioning, and VQA samples into the correct image-text input format for PaLI-Gemma 2.

In [ ]:

import re
import torch

# Clamp any out-of-range coords (grid is 0..1023)
_LOC_ANY_RE   = re.compile(r"<loc(\d{1,4})>")
_LOC1024_RE   = re.compile(r"<loc1024>")  # kept for backwards compatibility

def _infer_task_from_prefix(prefix: str) -> str:
    p = (prefix or "").strip().lower()
    if p.startswith("detect"):  return "detect"
    if p.startswith("segment"): return "segment"
    if p.startswith("caption"): return "caption"
    return "vqa"  # default: your VQA uses the question in prefix

def _sanitize_prefix(prefix: str, task: str) -> str:
    """
    Normalize task prefixes so the model always sees consistent prompts.
    Handles caption, detection, segmentation, and VQA.
    """
    prefix_raw = (prefix or "").strip()      # keep original for VQA
    p = prefix_raw.lower()                   # your data uses lowercased prompts

    if task == "caption":
        return "caption en"

    if task in ["detect", "segment"]:
        # ensure single terminal period
        p = re.sub(r"[.?!\s]+$", "", p)
        return p + "."

    if task == "vqa":
        # preserve original casing for the question
        pr = re.sub(r"\s+$", "", prefix_raw)
        return f"answer en {pr}"

    return p

def _sanitize_suffix(suffix: str) -> str:
    s = (suffix or "")
    # Robust clamp: any <locNNNN> with NNNN>1023 -> <loc1023>
    def _clamp(m):
        n = int(m.group(1))
        return f"<loc{min(n, 1023):04d}>"
    s = _LOC_ANY_RE.sub(_clamp, s)
    # (Keep label content as-is otherwise)
    return s

def collate_fn(examples):
    """
    Supports: detection, segmentation, captioning, VQA.
    Expects each example to have:
      ex["image"]  (PIL image),
      ex["prefix"] (str),
      ex["suffix"] (str)  # label for all tasks (incl. VQA answer/caption text)
    """
    texts, labels, images = [], [], []

    for ex in examples:
        prefix = ex.get("prefix", "")
        suffix = ex.get("suffix", "")

        # assert no empty labels since your dataset already normalized them
        if not isinstance(suffix, str) or suffix.strip() == "":
          suffix = "no buildings"

        task  = _infer_task_from_prefix(prefix)
        text  = _sanitize_prefix(prefix, task)
        label = _sanitize_suffix(suffix)

        img = ex["image"].convert("RGB")
        texts.append(f"<image>{text}")   # include <image> token
        labels.append(label)
        images.append(img)

    tokens = processor(
        text=texts,
        images=images,
        suffix=labels,          # loss computed ONLY on the suffix
        return_tensors="pt",
        padding="longest",
        truncation=False,
        max_length=1024          # text token budget
    )


    return tokens



### Define training arguments

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ALL IN_pg/New Work for Journal /Best Models/paligemma2_mix_ft_full",
    num_train_epochs=10,
    remove_unused_columns=False,
    per_device_train_batch_size=2,            # increase if memory allows
    gradient_accumulation_steps=8,            # aim for effective batch size ~16
    warmup_ratio=0.05,
    learning_rate=3e-5,
    weight_decay=0.01,                        # more standard, helps generalization
    adam_beta2=0.999,
    logging_steps=50,                         # log a bit less often
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,                       # keep 2 checkpoints
    fp16=True, bf16=False,
    report_to="wandb",
    run_name="paligemma2_mix_ft_full",
    dataloader_pin_memory=True,               # usually True improves speed
    eval_strategy="steps",
    eval_steps=500,                           # eval less often → faster training
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",          # custom: detection/segmentation F1 instead of loss
    greater_is_better=False,                   # for F1 higher is better
    seed=42,
    data_seed=42,
)


In [ ]:
# TRAIN_FRACTION = 0.25   # or 0.50
# SEED = getattr(args, "data_seed", 42)

# train_ds_small = stratified_subset_by_task(
#     train_ds,
#     fraction=TRAIN_FRACTION,
#     seed=SEED,
#     min_per_task=1,            # ensure at least one sample if that task exists
#     group_vqa_by_image=True,   # keeps all QAs for selected VQA images
# )

[StratifiedSubset] per-task counts (task, total_rows, chosen_rows):
  - caption                  total= 1564 -> chosen=  391  (25.0%)
  - detection                total= 1564 -> chosen=  391  (25.0%)
  - segmentation             total= 1564 -> chosen=  391  (25.0%)
  - vqa (grouped-by-image)   total= 7820 -> chosen= 1955  (25.0%)
[StratifiedSubset] total chosen = 3128 / 12512


In [ ]:
# before training, once:
# processor.tokenizer.padding_side = "right"
# processor.tokenizer.truncation_side = "right"

In [ ]:
from transformers import Trainer

trainer = Trainer(
        model=model,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate_fn,
        args=args,
        processing_class=processor,
        )

/tmp/ipython-input-2732322068.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()
RUN_DIR   = "/content/drive/MyDrive/ALL IN_pg/New Work for Journal /Best Models/paligemma2_mix_ft_full"
processor.save_pretrained(RUN_DIR)

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
500,1.851700,1.227494
1000,1.593800,1.182418
1500,1.414900,1.233777
2000,1.169300,1.388850
2500,0.986400,1.612532
3000,0.821700,1.787895
3500,0.679800,1.914355


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


In [ ]:
# RUN_DIR = "outputs/paligemma_multi"
# processor.save_pretrained(RUN_DIR)

In [ ]:
metrics = trainer.evaluate()   # uses the eval_dataset you passed to Trainer
print(metrics)

# (optional) a rough perplexity for text tokens
import math
if "eval_loss" in metrics:
    print("eval_ppl (approx):", math.exp(metrics["eval_loss"]))

{'eval_loss': 1.2361831665039062, 'eval_runtime': 26.6841, 'eval_samples_per_second': 18.288, 'eval_steps_per_second': 2.286, 'epoch': 4.0, 'num_input_tokens_seen': 0}
eval_ppl (approx): 3.4424491026117936


Evaluation **Below**